# Module 1 • Foundations of Natural Language Processing

# Lesson 3 • The Natural Language Processing Pipeline

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Beginner  
**Estimated study time:** 150–190 minutes

---

This notebook combines all five parts into one sequential executable lesson.

## Complete Lesson Map

- Part 1: Problem definition, success criteria, baselines, and risk
- Part 2: Data acquisition, governance, annotation, and splitting
- Part 3: Preprocessing, representation, modeling, and reproducibility
- Part 4: Metrics, error analysis, robustness, slices, and abstention
- Part 5: Deployment, monitoring, documentation, and improvement

## Table of Contents

1. From Model to System
2. The End-to-End Lifecycle
3. Problem Definition
4. Inputs, Outputs, and Boundaries
5. Success Criteria
6. Baselines
7. Risk Analysis
8. Iteration and Feedback
9. Part 1 Summary

# 1. From Model to System

A trained model is only one component of an NLP system.

```text
Users and Data Sources
          ↓
Input Validation and Processing
          ↓
Representation and Model
          ↓
Decision Logic and Post-processing
          ↓
Application Output
          ↓
Logging, Monitoring, and Feedback
```

A complete system must define how data enters, how invalid input is handled, how predictions become actions, when humans intervene, and how failures are detected.

> **Key Idea**
>
> The goal is not merely to train a model. The goal is to deliver a reliable language-processing capability for a defined use case.

# 2. The End-to-End Lifecycle

A practical NLP lifecycle includes:

1. problem definition;
2. data acquisition and governance;
3. inspection and annotation;
4. preprocessing and representation;
5. model or rule development;
6. evaluation and error analysis;
7. deployment and integration;
8. monitoring and maintenance;
9. feedback-driven improvement.

These stages interact. Evaluation may reveal annotation defects, deployment may impose latency limits, and monitoring may expose new vocabulary or changing user behavior.

In [ ]:
import pandas as pd

lifecycle = pd.DataFrame([
    ("Problem definition", "What decision or output is required?"),
    ("Data acquisition", "What data is permitted and representative?"),
    ("Inspection and annotation", "Are records and labels reliable?"),
    ("Preparation", "How should text be validated and represented?"),
    ("Modeling", "Which baseline and model family fit the constraints?"),
    ("Evaluation", "Does the system meet technical and operational goals?"),
    ("Deployment", "How does the model enter the application workflow?"),
    ("Monitoring", "How are drift, failures, and impact detected?"),
    ("Improvement", "Which evidence justifies the next change?"),
], columns=["Stage", "Primary question"])
lifecycle

# 3. Problem Definition

Requests such as *build a chatbot* or *analyze customer messages* are too broad. A usable problem definition specifies:

- language task and unit of analysis;
- input source and format;
- required output;
- target users and downstream action;
- supported languages and domains;
- unacceptable failures;
- latency, cost, privacy, and interpretability constraints;
- fallback behavior and evaluation criteria.

**Operational example:** Classify each incoming English support ticket as `account`, `billing`, `technical`, or `delivery`. Low-confidence predictions go to a human-review queue.

In [ ]:
from dataclasses import asdict, dataclass
from typing import Sequence

@dataclass
class NLPProblemDefinition:
    task: str
    input_unit: str
    output: str
    users: Sequence[str]
    downstream_action: str
    languages: Sequence[str]
    critical_failure: str
    fallback: str

problem = NLPProblemDefinition(
    task="multiclass text classification",
    input_unit="one support ticket",
    output="account, billing, technical, or delivery",
    users=["support operations"],
    downstream_action="route to a specialist queue",
    languages=["English"],
    critical_failure="routing to an unrelated team",
    fallback="human review for low confidence",
)
pd.Series(asdict(problem), name="Problem definition")

# 4. Inputs, Outputs, and Boundaries

The unit of analysis may be a token, sentence, message, document, conversation, document collection, or source-target pair. The output may be a label, score, extracted span, structured record, ranking, translation, summary, or generated response.

The automation boundary is equally important. A system may process only valid English single-intent tickets and defer unsupported languages, multiple intents, or uncertain cases.

# 5. Success Criteria

| Dimension | Example question |
|---|---|
| Predictive quality | Is macro F1 adequate for every class? |
| Coverage | What percentage can be processed automatically? |
| Reliability | How often is output invalid or unsupported? |
| Latency | Is the application response-time limit met? |
| Cost | Is inference affordable at expected volume? |
| Privacy | Is sensitive information minimized? |
| Fairness | Are performance differences across slices understood? |
| User impact | Does routing actually reduce handling time? |

Technical metrics must connect to operational outcomes.

# 6. Baselines

A baseline establishes the minimum reference point. Examples include majority-class prediction, keyword rules, the current manual process, TF-IDF with a linear classifier, or an existing production system.

A complex model is justified only when its improvement compensates for additional cost, latency, maintenance, and risk.

In [ ]:
baseline_options = pd.DataFrame([
    ("Majority class", "Very low", "Checks class imbalance"),
    ("Keyword rules", "Low", "Provides transparent behavior"),
    ("TF-IDF + linear model", "Moderate", "Strong classical baseline"),
    ("Pretrained Transformer", "High", "Tests contextual transfer"),
    ("LLM prompt", "Variable", "Tests flexible instruction behavior"),
], columns=["Approach", "Complexity", "Purpose"])
baseline_options

# 7. Risk Analysis

Identify who is affected, the cost of false positives and false negatives, sensitive data, misuse possibilities, conditions requiring review, and recovery procedures. A support-routing error is usually reversible; medical, legal, financial, or safety-critical workflows require stronger controls and narrower automation.

# 8. Iteration and Feedback

```text
Define → Build → Evaluate → Deploy → Monitor
  ↑         ↓         ↓          ↓
  └──── Refine data, labels, model, and requirements ────┘
```

More model complexity will not repair an incorrect task definition, poor labels, leakage, or a mismatch between training and deployment data.

# 9. Part 1 Summary

- A model is one component of a system.
- Problem definition establishes task, users, outputs, constraints, and failure costs.
- Success includes predictive and operational criteria.
- Baselines test whether complexity is justified.
- Risk analysis determines review and fallback requirements.
- NLP development is iterative and evidence-driven.

**Continue with Part 2:** data acquisition, governance, inspection, annotation, and splitting.

## Table of Contents

10. Data Acquisition and Provenance
11. Governance, Privacy, and Licensing
12. Dataset Inspection
13. Duplicate and Missing-Value Analysis
14. Annotation Quality
15. Representativeness and Imbalance
16. Data Leakage
17. Dataset Splitting
18. Part 2 Summary

# 10. Data Acquisition and Provenance

Text may come from internal logs, support conversations, surveys, public corpora, licensed datasets, websites, APIs, documents, databases, manual creation, or synthetic generation.

Document where the data came from, when it was collected, under which permission, which transformations were applied, and what population or domain it represents.

# 11. Governance, Privacy, and Licensing

Data availability does not automatically imply permission. Confirm licensing, consent, privacy, access control, retention, redistribution, and whether sensitive fields can be minimized or removed. Governance decisions should be recorded before modeling.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILENAME = "lesson_03_support_tickets.csv"
candidates = [
    Path("../../datasets/raw") / DATA_FILENAME,
    Path("datasets/raw") / DATA_FILENAME,
    Path("../datasets/raw") / DATA_FILENAME,
    Path(DATA_FILENAME),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Place lesson_03_support_tickets.csv in datasets/raw/ or beside the notebook."
    )
tickets = pd.read_csv(data_path, parse_dates=["created_at"])
print(f"Loaded {len(tickets)} rows from {data_path}")
tickets.head()

# 12. Dataset Inspection

Initial inspection verifies row and column counts, types, required fields, empty text, duplicates, labels, class distribution, language coverage, dates, and suspicious identifiers.

In [ ]:
print("Shape:", tickets.shape)
print("\nData types:")
print(tickets.dtypes)
print("\nMissing values:")
print(tickets.isna().sum())
print("\nLabel counts:")
print(tickets["label"].value_counts())
print("\nDate range:", tickets["created_at"].min(), "to", tickets["created_at"].max())

In [ ]:
import matplotlib.pyplot as plt
label_counts = tickets["label"].value_counts().sort_index()
plt.figure(figsize=(8, 4))
plt.bar(label_counts.index, label_counts.values)
plt.title("Support Ticket Label Distribution")
plt.xlabel("Label")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

# 13. Duplicate and Missing-Value Analysis

Duplicate text may distort class frequencies and leak copies across partitions. Missing metadata may be acceptable, but missing text usually prevents language analysis.

In [ ]:
duplicate_mask = tickets.duplicated(subset=["text"], keep=False)
print("Rows with duplicated text:")
display(tickets.loc[duplicate_mask, ["ticket_id", "text", "label"]])
print("\nRows with missing channel:")
display(tickets.loc[tickets["channel"].isna()])

In [ ]:
cleaned_tickets = (
    tickets.dropna(subset=["text", "label"])
    .drop_duplicates(subset=["text"], keep="first")
    .copy()
)
cleaned_tickets["channel"] = cleaned_tickets["channel"].fillna("unknown")
print("Original rows:", len(tickets))
print("Clean rows:", len(cleaned_tickets))

# 14. Annotation Quality

Guidelines should define each label, positive and negative examples, ambiguous cases, multi-intent handling, exclusions, required context, and escalation. Frequent disagreement may indicate an unclear task, insufficient context, or the need for multi-label annotation.

- `billing`: charges, payments, invoices, refunds, receipts, plans;
- `account`: access, profile, verification, credentials, security;
- `technical`: software behavior, errors, integrations, performance;
- `delivery`: shipping, tracking, courier, address, package condition.

In [ ]:
guidelines = {
    "account": "Access, profile, credentials, verification, or security",
    "billing": "Payment, charge, invoice, refund, receipt, or subscription",
    "technical": "Software error, malfunction, performance, or integration",
    "delivery": "Shipping, tracking, courier, address, or package",
}
pd.Series(guidelines, name="Guideline")

# 15. Representativeness and Imbalance

Production data may differ by topic, language, dialect, style, length, device, channel, season, and terminology. Sampling should reflect deployment rather than only the easiest available examples.

# 16. Data Leakage

Leakage occurs when training uses information unavailable for a genuine future prediction. Examples include duplicate messages across splits, labels embedded in text, post-outcome metadata, fitting preprocessing on all data, related conversation messages in different partitions, or training on future records and testing on the past.

# 17. Dataset Splitting

Training data fits parameters. Validation data supports design and threshold selection. Test data estimates final performance after choices are fixed. Stratification approximately preserves class proportions; grouped or temporal splitting may be required for related or time-dependent data.

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(
    cleaned_tickets,
    test_size=0.25,
    random_state=42,
    stratify=cleaned_tickets["label"],
)
print("Training rows:", len(train_data))
print("Test rows:", len(test_data))
pd.concat([
    train_data["label"].value_counts().rename("train"),
    test_data["label"].value_counts().rename("test"),
], axis=1).fillna(0).astype(int)

# 18. Part 2 Summary

- Data requires provenance, permission, governance, and representative sampling.
- Inspection occurs before preprocessing and modeling.
- Duplicates, missing fields, imbalance, and incorrect labels can invalidate evaluation.
- Annotation guidelines convert vague concepts into consistent targets.
- Leakage produces optimistic results without real generalization.
- Splitting strategy must match classes, groups, time, and deployment.

**Continue with Part 3:** preprocessing, representation, baseline construction, and reproducibility.

## Table of Contents

19. Preprocessing as a Design Decision
20. Conservative Normalization
21. Text Representation
22. Model Selection
23. Leakage-Safe Pipelines
24. Training and Inference
25. Feature Inspection
26. Reproducibility
27. Part 3 Summary

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILENAME = "lesson_03_support_tickets.csv"
candidates = [
    Path("../../datasets/raw") / DATA_FILENAME,
    Path("datasets/raw") / DATA_FILENAME,
    Path("../datasets/raw") / DATA_FILENAME,
    Path(DATA_FILENAME),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Place lesson_03_support_tickets.csv in datasets/raw/ or beside the notebook."
    )
tickets = pd.read_csv(data_path, parse_dates=["created_at"])
print(f"Loaded {len(tickets)} rows from {data_path}")
tickets.head()

In [ ]:
modeling_data = tickets.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).copy()
print(modeling_data.shape)
modeling_data.head()

# 19. Preprocessing as a Design Decision

Preprocessing may include validation, Unicode normalization, case handling, identifier masking, whitespace normalization, tokenization, stemming, lemmatization, or language-specific processing.

More preprocessing is not automatically better. Negation may be essential, punctuation may carry structure or emotion, case may identify names, and Transformer tokenizers usually expect text close to pretraining conditions.

# 20. Conservative Normalization

The function below applies Unicode compatibility normalization, lowercasing, URL and email placeholders, and whitespace normalization. It deliberately retains punctuation and function words.

In [ ]:
import re
import unicodedata

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b")
WHITESPACE_PATTERN = re.compile(r"\s+")

def normalize_text(value: str) -> str:
    """Apply conservative task-aware normalization."""
    value = unicodedata.normalize("NFKC", str(value))
    value = EMAIL_PATTERN.sub(" EMAIL_ADDRESS ", value)
    value = URL_PATTERN.sub(" URL ", value)
    value = value.lower()
    return WHITESPACE_PATTERN.sub(" ", value).strip()

In [ ]:
examples = [
    "  My APP crashes after login!!!  ",
    "Contact help@example.com for details.",
    "See https://example.com/status for updates.",
]
for item in examples:
    print("Before:", repr(item))
    print("After: ", repr(normalize_text(item)))
    print()

# 21. Text Representation

| Representation | Typical role |
|---|---|
| Rules and lexicons | Narrow transparent systems |
| Bag of Words | Classical classification |
| TF-IDF | Strong sparse baseline |
| Static embeddings | Dense lexical representation |
| Contextual embeddings | Transfer learning and semantic tasks |
| Model token IDs | Transformer training and inference |
| Retrieved documents | Grounded generative systems |

This lesson uses TF-IDF because it is efficient and suitable as a baseline.

# 22. Model Selection

Consider data volume, output type, context length, languages, hardware, latency, transparency, update frequency, domain shift, and deployment environment. The simplest model satisfying the actual requirements is usually the best starting point.

# 23. Leakage-Safe Pipelines

A scikit-learn `Pipeline` fits TF-IDF only on training text and applies the same fitted transformation during evaluation and inference.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    modeling_data["text"], modeling_data["label"],
    test_size=0.25, random_state=42, stratify=modeling_data["label"]
)
text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=normalize_text, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
text_pipeline

# 24. Training and Inference

`fit` estimates the vocabulary, feature weights, and classifier parameters from the training partition only.

In [ ]:
text_pipeline.fit(X_train, y_train)
predictions = text_pipeline.predict(X_test)
probabilities = text_pipeline.predict_proba(X_test)
pd.DataFrame({
    "text": X_test.to_numpy(),
    "actual": y_test.to_numpy(),
    "predicted": predictions,
    "confidence": probabilities.max(axis=1),
}).sort_values("confidence")

# 25. Feature Inspection

Linear coefficients can show which features most strongly support each class in this fitted dataset. They are diagnostics, not complete causal explanations.

In [ ]:
import numpy as np
vectorizer = text_pipeline.named_steps["tfidf"]
classifier = text_pipeline.named_steps["classifier"]
features = vectorizer.get_feature_names_out()
result = {}
for index, class_name in enumerate(classifier.classes_):
    top = np.argsort(classifier.coef_[index])[-8:][::-1]
    result[class_name] = features[top].tolist()
pd.DataFrame({k: pd.Series(v) for k, v in result.items()})

# 26. Reproducibility

Record data version, preprocessing code, annotation version, split method, random seed, library versions, parameters, evaluation script, environment, and saved artifact identifiers. Seeds improve repeatability but do not guarantee identical results across every platform.

In [ ]:
import platform
import sklearn
pd.Series({
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "random_state": 42,
    "training_rows": len(X_train),
    "test_rows": len(X_test),
}, name="Run metadata")

# 27. Part 3 Summary

- Preprocessing must preserve task-relevant information.
- Representation converts text into algorithm inputs.
- TF-IDF plus a linear classifier is a strong baseline.
- Unified pipelines keep transformations consistent and leakage-safe.
- Feature inspection supports debugging but is not a complete explanation.
- Reproducibility requires data, code, parameters, environment, and artifacts.

**Continue with Part 4:** metrics, error analysis, robustness, slices, and abstention.

## Table of Contents

28. Evaluation Questions
29. Metrics
30. Confusion Matrix
31. Error Analysis
32. Cross-Validation
33. Robustness Probes
34. Slice Evaluation
35. Thresholds and Abstention
36. Part 4 Summary

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILENAME = "lesson_03_support_tickets.csv"
candidates = [
    Path("../../datasets/raw") / DATA_FILENAME,
    Path("datasets/raw") / DATA_FILENAME,
    Path("../datasets/raw") / DATA_FILENAME,
    Path(DATA_FILENAME),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Place lesson_03_support_tickets.csv in datasets/raw/ or beside the notebook."
    )
tickets = pd.read_csv(data_path, parse_dates=["created_at"])
print(f"Loaded {len(tickets)} rows from {data_path}")
tickets.head()

In [ ]:
import re
import unicodedata

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b")
WHITESPACE_PATTERN = re.compile(r"\s+")

def normalize_text(value: str) -> str:
    """Apply conservative task-aware normalization."""
    value = unicodedata.normalize("NFKC", str(value))
    value = EMAIL_PATTERN.sub(" EMAIL_ADDRESS ", value)
    value = URL_PATTERN.sub(" URL ", value)
    value = value.lower()
    return WHITESPACE_PATTERN.sub(" ", value).strip()

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

evaluation_data = tickets.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).copy()
X_train, X_test, y_train, y_test = train_test_split(
    evaluation_data["text"], evaluation_data["label"], test_size=0.25,
    random_state=42, stratify=evaluation_data["label"]
)
text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=normalize_text, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
text_pipeline.fit(X_train, y_train)
y_pred = text_pipeline.predict(X_test)
y_proba = text_pipeline.predict_proba(X_test)

# 28. Evaluation Questions

Ask which classes fail, which errors are costly, whether performance is stable across channels and styles, whether confidence supports deferral, whether test data reflects deployment, and whether the model beats the baseline.

# 29. Metrics

\[
\text{Precision}=\frac{TP}{TP+FP},\qquad
\text{Recall}=\frac{TP}{TP+FN}
\]

\[
F_1=2\cdot\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}
\]

Macro averaging gives each class equal weight. Weighted averaging uses class support. Accuracy is the total proportion correct.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
pd.DataFrame(classification_report(y_test, y_pred, output_dict=True, zero_division=0)).T

The dataset is small and synthetic. Its metrics demonstrate the workflow and are not production claims.

# 30. Confusion Matrix

A confusion matrix shows which actual classes are assigned to which predicted classes.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    labels=text_pipeline.named_steps["classifier"].classes_,
    xticks_rotation=45,
)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# 31. Error Analysis

A useful error table includes original text, actual label, prediction, confidence, error type, likely cause, and proposed correction. Wrong high-confidence predictions deserve special attention.

In [ ]:
errors = pd.DataFrame({
    "text": X_test.to_numpy(),
    "actual": y_test.to_numpy(),
    "predicted": y_pred,
    "confidence": y_proba.max(axis=1),
})
errors["correct"] = errors["actual"] == errors["predicted"]
errors.sort_values(["correct", "confidence"])

# 32. Cross-Validation

One split may be sensitive to the selected examples. Stratified cross-validation repeats evaluation across several partitions.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
scores = cross_val_score(
    clone(text_pipeline), evaluation_data["text"], evaluation_data["label"],
    cv=cv, scoring="f1_macro"
)
print("Fold macro F1:", np.round(scores, 3))
print(f"Mean: {scores.mean():.3f}")
print(f"Standard deviation: {scores.std():.3f}")

# 33. Robustness Probes

Robustness probes change form while preserving intended meaning, or intentionally test unsupported conditions such as multiple intents and unsupported languages.

In [ ]:
robustness = pd.DataFrame([
    ("CARD CHARGED TWICE!!!", "billing"),
    ("cant login password forgotten", "account"),
    ("app freezes... again", "technical"),
    ("tracking unchanged package late", "delivery"),
    ("I cannot log in and I was also charged twice", "multiple_intents"),
    ("الفاتورة لم تصل", "unsupported_language"),
], columns=["text", "expected_or_condition"])
robustness["prediction"] = text_pipeline.predict(robustness["text"])
robustness["confidence"] = text_pipeline.predict_proba(robustness["text"]).max(axis=1)
robustness

The final two inputs violate the original task boundary. A reliable system should detect or defer them rather than treating every input as standard English single-label text.

# 34. Slice Evaluation

Aggregate results may hide weak performance by channel, language, length, product, or population. The example below reports out-of-fold accuracy by channel.

In [ ]:
from sklearn.model_selection import cross_val_predict
oof = cross_val_predict(clone(text_pipeline), evaluation_data["text"], evaluation_data["label"], cv=cv)
slices = evaluation_data[["channel", "label"]].copy()
slices["prediction"] = oof
slices["correct"] = slices["label"] == slices["prediction"]
slices.groupby("channel", dropna=False).agg(
    examples=("correct", "size"), accuracy=("correct", "mean")
).sort_values("examples", ascending=False)

Slice results from tiny samples are unstable. Production reports should include support and uncertainty.

# 35. Thresholds and Abstention

A system need not force a decision for every input. High confidence may permit automatic routing, while low confidence, unsupported language, invalid input, or multiple intents should trigger review or redirection.

In [ ]:
threshold = 0.55
threshold_table = errors.copy()
threshold_table["decision"] = np.where(
    threshold_table["confidence"] >= threshold,
    threshold_table["predicted"],
    "human_review",
)
threshold_table.sort_values("confidence")

# 36. Part 4 Summary

- Evaluation must reflect classes, users, and failure costs.
- Accuracy can hide class-level weaknesses.
- Confusion and example analysis reveal error structure.
- Cross-validation estimates partition sensitivity.
- Robustness probes test behavior beyond clean examples.
- Slice evaluation checks hidden subgroup weakness.
- Abstention limits automation when confidence or validity is low.

**Continue with Part 5:** deployment, monitoring, documentation, feedback, and exercises.

## Table of Contents

37. Deployment Patterns
38. Inference Contracts
39. Batch Inference
40. Logging and Observability
41. Monitoring and Drift
42. Versioning and Documentation
43. Human Feedback
44. Production Readiness Checklist
45. Knowledge Check
46. Exercises
47. Challenge Exercises
48. Summary
49. Further Reading
50. Next Lesson

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILENAME = "lesson_03_support_tickets.csv"
candidates = [
    Path("../../datasets/raw") / DATA_FILENAME,
    Path("datasets/raw") / DATA_FILENAME,
    Path("../datasets/raw") / DATA_FILENAME,
    Path(DATA_FILENAME),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Place lesson_03_support_tickets.csv in datasets/raw/ or beside the notebook."
    )
tickets = pd.read_csv(data_path, parse_dates=["created_at"])
print(f"Loaded {len(tickets)} rows from {data_path}")
tickets.head()

In [ ]:
import re
import unicodedata

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b")
WHITESPACE_PATTERN = re.compile(r"\s+")

def normalize_text(value: str) -> str:
    """Apply conservative task-aware normalization."""
    value = unicodedata.normalize("NFKC", str(value))
    value = EMAIL_PATTERN.sub(" EMAIL_ADDRESS ", value)
    value = URL_PATTERN.sub(" URL ", value)
    value = value.lower()
    return WHITESPACE_PATTERN.sub(" ", value).strip()

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

deployment_data = tickets.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).copy()
production_candidate = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=normalize_text, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
# Evaluation was demonstrated in Part 4; this final teaching candidate uses all cleaned rows.
production_candidate.fit(deployment_data["text"], deployment_data["label"])

# 37. Deployment Patterns

| Pattern | Description | Typical use |
|---|---|---|
| Batch | Process stored documents together | Reports and archives |
| Online service | Predict for each request | Web and mobile applications |
| Streaming | Process continuous events | Queues and monitoring |
| On-device | Run near the user | Privacy or latency constraints |
| Human-in-the-loop | Combine model and review | Uncertain or high-impact cases |

Architecture should match volume, latency, privacy, reliability, and update requirements.

# 38. Inference Contracts

An inference contract defines accepted inputs and guaranteed output fields. Validation should reject non-string, empty, excessively long, malformed, or unsupported inputs before inference.

In [ ]:
def validate_ticket_text(value: object, max_characters: int = 5000) -> str:
    if not isinstance(value, str):
        raise TypeError("Ticket text must be a string.")
    value = value.strip()
    if not value:
        raise ValueError("Ticket text must not be empty.")
    if len(value) > max_characters:
        raise ValueError(f"Ticket text exceeds {max_characters} characters.")
    return value


def predict_ticket(text: object, confidence_threshold: float = 0.55) -> dict:
    text = validate_ticket_text(text)
    probabilities = production_candidate.predict_proba([text])[0]
    classes = production_candidate.named_steps["classifier"].classes_
    index = int(np.argmax(probabilities))
    label = str(classes[index])
    confidence = float(probabilities[index])
    return {
        "predicted_label": label,
        "confidence": round(confidence, 4),
        "decision": label if confidence >= confidence_threshold else "human_review",
    }

predict_ticket("My invoice shows an unexpected charge.")

# 39. Batch Inference

Batch processing applies the same validated interface to a collection of records.

In [ ]:
incoming = pd.DataFrame([
    ("NEW-001", "My password reset link does not work."),
    ("NEW-002", "The courier has not delivered my package."),
    ("NEW-003", "The app shows an error whenever I upload a file."),
    ("NEW-004", "I need a copy of my last invoice."),
    ("NEW-005", "I have several unrelated questions."),
], columns=["ticket_id", "text"])
outputs = []
for record in incoming.to_dict(orient="records"):
    outputs.append({**record, **predict_ticket(record["text"])})
pd.DataFrame(outputs)

# 40. Logging and Observability

Useful signals include request time, latency, model and preprocessing version, input length, detected language, predicted label, confidence, fallback status, validation failure category, system errors, and later human correction.

Raw user text should not be logged by default when metadata or secure sampled review is sufficient.

# 41. Monitoring and Drift

**Data drift** means the input distribution changed: message length, channels, terminology, languages, or topic frequencies. **Concept drift** means the relationship between text and the correct output changed: routing policy, category definitions, or product behavior.

Monitoring combines automatic statistics, sampled human review, downstream outcomes, and incident reporting.

In [ ]:
reference = deployment_data["text"]
current = pd.Series([
    "APP BROKEN",
    "invoice invoice invoice invoice",
    "tracking unchanged since last week",
    "cannot authenticate after security policy update",
    "new premium bundle renewal produced an unexpected adjustment",
])
pd.DataFrame({
    "reference": {
        "examples": len(reference),
        "mean_characters": reference.str.len().mean(),
        "mean_words": reference.str.split().str.len().mean(),
    },
    "current_batch": {
        "examples": len(current),
        "mean_characters": current.str.len().mean(),
        "mean_words": current.str.split().str.len().mean(),
    },
})

This is only an illustration. Production monitoring requires historical baselines, thresholds, sample-size rules, and investigation procedures.

# 42. Versioning and Documentation

Associate each release with model version, code commit, training-data snapshot, annotation version, preprocessing version, parameters, evaluation report, intended and prohibited uses, limitations, owner, and rollback procedure.

In [ ]:
model_card = {
    "model_name": "lesson-03-support-ticket-router",
    "version": "0.1.0-demo",
    "task": "four-class English ticket routing",
    "model_family": "TF-IDF + logistic regression",
    "training_examples": len(deployment_data),
    "labels": sorted(deployment_data["label"].unique().tolist()),
    "intended_use": "education and baseline development",
    "not_for": "high-impact decisions or unsupported languages",
    "limitations": ["small synthetic data", "single-label output", "uncalibrated confidence"],
}
pd.Series(model_card, name="Model card")

# 43. Human Feedback

Corrections, override reasons, unsupported categories, repeated failures, complaints, and incidents can guide improvement. Feedback is not automatically trustworthy training data; it must be validated, de-duplicated, reviewed for privacy, and sampled fairly.

# 44. Production Readiness Checklist

## Problem and Governance
- [ ] Task and automation boundary are documented.
- [ ] Data use, privacy, retention, and licensing are approved.
- [ ] Failure costs and human fallback are defined.

## Data and Modeling
- [ ] Provenance and annotation guidelines are versioned.
- [ ] Leakage checks are complete.
- [ ] Baseline comparisons are reported.
- [ ] Evaluation includes classes, slices, and robustness cases.

## Deployment and Monitoring
- [ ] Input and output contracts are validated.
- [ ] Model, code, and preprocessing versions are recorded.
- [ ] Latency, capacity, rollback, and failure behavior are tested.
- [ ] Drift indicators, review cadence, and ownership are assigned.

# 45. Knowledge Check

1. Why is a model not a complete NLP system?
2. Which elements belong in a problem definition?
3. Why should success include more than accuracy?
4. What is the purpose of a baseline?
5. What should provenance record?
6. How can duplicates cause leakage?
7. Why is annotation disagreement useful evidence?
8. What is the difference between preprocessing and representation?
9. Why fit transformations only on training data?
10. How do macro and weighted F1 differ?
11. What can robustness probes reveal?
12. How do data drift and concept drift differ?
13. Why is abstention useful?
14. Which artifacts support release reproduction?

# 46. Exercises

## Exercise 1 — Problem Contract
Write a complete task definition for one NLP application.

## Exercise 2 — Dataset Audit
Report empty strings, unexpected labels, and unusually short or long messages.

## Exercise 3 — Annotation Guideline
Add a `cancellation` label with inclusion, exclusion, positive, and ambiguous examples.

## Exercise 4 — Three-Way Split
Create train, validation, and test sets while preserving class proportions.

## Exercise 5 — Preprocessing Comparison
Compare conservative normalization with punctuation removal and explain lost information.

## Exercise 6 — Baseline Comparison
Compare logistic regression with Multinomial Naïve Bayes using identical folds and macro F1.

## Exercise 7 — Error Taxonomy
Define categories for wrong class, unsupported language, multiple intents, empty input, and low confidence.

## Exercise 8 — Monitoring Specification
Define five production indicators, expected ranges, and response actions.

# 47. Challenge Exercises

## Challenge 1 — Group-Aware Splitting
Prevent related conversation messages from entering different partitions.

## Challenge 2 — Confidence Calibration
Compare raw and calibrated probabilities before choosing an abstention threshold.

## Challenge 3 — Multi-Intent Design
Redesign the task as multi-label classification and revise annotation, metrics, and output.

## Challenge 4 — Multilingual Boundary
Add Arabic tickets and document required representation and evaluation changes.

## Challenge 5 — Deployment Architecture
Design an online service with validation, model service, human-review queue, monitoring, registry, and rollback.

# 48. Summary

- Precise problem definition establishes the task and automation boundary.
- Data requires provenance, governance, inspection, annotation, and appropriate splitting.
- Preprocessing and representation are task-dependent.
- Reproducible pipelines keep transformations consistent.
- Evaluation combines metrics, errors, robustness, slices, and failure costs.
- Deployment requires contracts, validation, logging, versioning, and fallback.
- Monitoring detects drift and changing requirements.
- Human feedback supports improvement only after quality and governance controls.

# 49. Further Reading and References

- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Eisenstein, J. (2019). *Introduction to Natural Language Processing*.
- Gebru, T., et al. (2021). *Datasheets for Datasets*.
- Mitchell, M., et al. (2019). *Model Cards for Model Reporting*.
- Bender, E. M., & Friedman, B. (2018). *Data Statements for NLP*.
- Sculley, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*.
- scikit-learn documentation for pipelines, metrics, model selection, and calibration.
- NIST AI Risk Management Framework.

# 50. Next Lesson

## Lesson 4 • The Python Ecosystem for Natural Language Processing

The next lesson introduces Python text processing, NumPy, Pandas, Matplotlib, scikit-learn, NLTK, spaCy, and the Hugging Face ecosystem. It will cover environment management, reproducible notebooks, library roles, and tool selection for each pipeline stage.